[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Tuning &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from scipy.stats import loguniform, randint, uniform
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, GroupKFold, KFold, RandomizedSearchCV, cross_val_score

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", train["station"].nunique(), "training stations,",
      len(train), "rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 300 training stations, 15000 rows


The worked examples' tables and folds, rebuilt here from the same made-up stations so that each answer
runs on its own.

In [2]:
X, y, stations = train[FEATURES], train["tmin"], train["station"]
folds = GroupKFold(n_splits=5, shuffle=True, random_state=0)

**1.** Cross-validate an `XGBRegressor` with 200 trees and a learning rate of 0.1 on the station folds, and print each fold's mean squared error and their mean.

In [3]:
model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=0)
scores = -cross_val_score(model, X, y, groups=stations, cv=folds, scoring="neg_mean_squared_error")
print("each fold:", scores.round(3), "| mean", round(scores.mean(), 3), "| spread", round(scores.std(), 3))

lightgbm_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.1, random_state=0, verbose=-1)
lightgbm_scores = -cross_val_score(lightgbm_model, X, y, groups=stations, cv=folds,
                                   scoring="neg_mean_squared_error")
print("XGBoost minus LightGBM:", (scores - lightgbm_scores).round(3))

each fold: [2.824 2.511 2.767 2.622 3.404] | mean 2.825 | spread 0.309
XGBoost minus LightGBM: [ 0.093  0.023  0.018 -0.01   0.041]


XGBoost's folds score from 2.511 to 3.404, a mean of 2.825, beside LightGBM's 2.792 with the same number of trees and learning rate. XGBoost's loss is higher on four of the five folds, by 0.018 to 0.093, and lower on fold 4 by 0.010. The two models also differ in tree shape and penalty, so this one comparison ranks neither library.

**2.** Run a grid search for LightGBM over `num_leaves` of 15 and 31 and `learning_rate` of 0.05 and 0.1, with 200 trees, and print the chosen settings and their mean score. Name it `rate_grid`, so that `grid` still holds the worked examples' search.

In [4]:
rate_grid = GridSearchCV(lgb.LGBMRegressor(n_estimators=200, random_state=0, verbose=-1),
                         {"num_leaves": [15, 31], "learning_rate": [0.05, 0.1]},
                         cv=folds, scoring="neg_mean_squared_error")
rate_grid.fit(X, y, groups=stations)
print(rate_grid.best_params_, round(-rate_grid.best_score_, 3))
print((-rate_grid.cv_results_["mean_test_score"]).round(3), "for", rate_grid.cv_results_["params"])

{'learning_rate': 0.05, 'num_leaves': 15} 2.739
[2.739 2.746 2.748 2.792] for [{'learning_rate': 0.05, 'num_leaves': 15}, {'learning_rate': 0.05, 'num_leaves': 31}, {'learning_rate': 0.1, 'num_leaves': 15}, {'learning_rate': 0.1, 'num_leaves': 31}]


The grid chooses 15 leaves at a learning rate of 0.05, at 2.739. The four candidates lie between 2.739 and 2.792, 0.053 apart, and the two at 0.05 are 0.007 apart, closer than the pair that the worked examples showed trading places from fold to fold, so the choice is a slight preference rather than a clear winner.

**3.** Run a random search of 6 candidates for LightGBM over `num_leaves` from `randint(4, 128)` and `min_child_samples` from `randint(5, 100)`, with `random_state=0`, and print the candidates sorted by their mean score.

In [5]:
search = RandomizedSearchCV(lgb.LGBMRegressor(n_estimators=200, learning_rate=0.1, random_state=0, verbose=-1),
                            {"num_leaves": randint(4, 128), "min_child_samples": randint(5, 100)},
                            n_iter=6, cv=folds, scoring="neg_mean_squared_error", random_state=0)
search.fit(X, y, groups=stations)
drawn = pd.DataFrame(search.cv_results_)
table = pd.DataFrame({"num_leaves": drawn["param_num_leaves"], "min_child_samples": drawn["param_min_child_samples"],
                      "mean": -drawn["mean_test_score"], "spread": drawn["std_test_score"]})
print(table.sort_values("mean", kind="stable").round(3).to_string(index=False))

 num_leaves  min_child_samples  mean  spread
         51                 49 2.782   0.283
         71                 69 2.798   0.263
        107                 72 2.848   0.263
         91                 41 2.876   0.283
         87                 14 2.901   0.333
        118                 26 2.921   0.325


The six draws score from 2.782 to 2.921, and none reaches the worked grid's 2.723. A random search tries only the values it happens to draw, and these six drew no leaf count below 51, far from the 7 the grid chose.

**4.** Run `lgb.cv` on the station folds with a learning rate of 0.05 in place of 0.1, and print how many rounds it keeps and its mean score.

In [6]:
slower = lgb.cv(
    {"objective": "regression", "learning_rate": 0.05, "num_leaves": 31, "seed": 0, "verbose": -1},
    lgb.Dataset(X, y), num_boost_round=2000, folds=list(folds.split(X, y, groups=stations)),
    callbacks=[lgb.early_stopping(50, verbose=False)],
)
print(len(slower["valid l2-mean"]), "rounds | mean", round(slower["valid l2-mean"][-1], 3))

433 rounds | mean 2.709


At 0.05 the folds keep 433 rounds and reach 2.709, against 150 rounds and 2.764 at 0.1. Halving the learning rate took about 2.9 times as many trees for a slightly lower score, in this one run.

**5.** Score the grid's chosen model, `grid.best_estimator_`, and an `LGBMRegressor` with its default settings on the validation stations, which took no part in the search.

In [7]:
grid = GridSearchCV(lgb.LGBMRegressor(n_estimators=200, learning_rate=0.1, random_state=0, verbose=-1),
                    {"num_leaves": [7, 31, 127], "min_child_samples": [5, 20, 80]},
                    cv=folds, scoring="neg_mean_squared_error")
grid.fit(X, y, groups=stations)
default = lgb.LGBMRegressor(random_state=0, verbose=-1).fit(X, y)

for name, chosen in (("chosen by the grid", grid.best_estimator_), ("default settings", default)):
    print(name, round(mean_squared_error(valid["tmin"], chosen.predict(valid[FEATURES])), 3))

chosen by the grid 2.483
default settings 2.553


On the validation stations the grid's choice scores 2.483 and the default settings, 100 trees with 31 leaves and a smallest leaf of 20, score 2.553. The search never saw these stations. On them its choice beats the default settings, which were not among its nine candidates, so this compares the search with no search at all, on one small set of 50 stations.

---

&#8592; **Back to:** [Tuning](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/12-tuning.ipynb)  &nbsp;&middot;&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)
